# Link agentic share of spend at the top agentic merchants

**Question (team review, 2026-10-07, item 12).** The dashboard counts agentic checkout only where Stripe issues a Link virtual card (`LINKAGNT*` / `LINK*`), i.e. at merchants that do not take Stripe directly. How large is that Link-agent volume relative to each top merchant's total card volume in the panel, before and after the Muse launch?

**Method**
- Merchants: the top agentic merchants in the app's trailing-28-day table (eBay, Walmart.com, American Airlines, Porkbun, Etsy) plus GoDaddy, which the team named. Square (`SQ *`) is left out: it is a processor covering many sellers, not one merchant.
- Rows: every Yodlee F3/F4/F6 card + bank row since 2026-08-01 whose descriptor names the merchant, one pruned range query per source, aggregated in Snowflake.
- Classes: **Link agent** (the app's agentic rule), **Link wallet** (`LINK.COM*`, the ordinary Stripe Link wallet, so Stripe-processed but not an agent), **other** (any other descriptor; the processor is not visible).
- Limitation: a purchase an agent makes through a merchant's own Stripe checkout carries the merchant's normal descriptor, so it lands in **other** and cannot be told apart from human purchases. The Link-agent share is therefore a floor on agent activity at each merchant, not a measure of it.

In [1]:
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import snowflake.connector

APP = Path.cwd().parent / "app"
sys.path.insert(0, str(APP))
import sections as S

pd.set_option("display.width", 200)
conn = snowflake.connector.connect(connection_name="default")


def q(sql: str) -> pd.DataFrame:
    df = conn.cursor().execute(sql).fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df


START, CARD_THROUGH, LAUNCH = "2026-08-01", "2026-10-03", "2026-09-08"
MERCHANTS = {  # label -> descriptor pattern
    "eBay": "%ebay%",
    "Walmart.com": "%walmart.com%",
    "American Airlines": "%american air%",
    "Porkbun": "%porkbun%",
    "Etsy": "%etsy%",
    "GoDaddy": "%godaddy%",
}

Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...


In [2]:
def merchant_sql(src: dict) -> str:
    d = src["desc"]
    key = "CASE " + " ".join(f"WHEN {d} ILIKE '{p}' THEN '{m}'" for m, p in MERCHANTS.items()) + " END"
    return f"""
        SELECT '{src['name']}' AS source, {key} AS merchant, {src['date']}::DATE AS day,
               CASE WHEN {S.agentic_sql(d)} THEN 'Link agent'
                    WHEN {d} ILIKE '%link.com*%' THEN 'Link wallet' ELSE 'other' END AS cls,
               COUNT(DISTINCT {src['txn']}) AS txns, SUM({src['amt']}) AS spend
        FROM {src['table']}
        WHERE {S._window(src, START, CARD_THROUGH)}
          AND ({" OR ".join(f"{d} ILIKE '{p}'" for p in MERCHANTS.values())})
          AND {src['amt']} > 0
        GROUP BY 1, 2, 3, 4
    """

with ThreadPoolExecutor(6) as pool:
    agg = pd.concat(pool.map(lambda s: q(merchant_sql(s)), S.CARD_SOURCES), ignore_index=True)
agg["spend"] = agg.spend.astype(float)
agg["period"] = (pd.to_datetime(agg.day) >= LAUNCH).map({False: "pre-launch", True: "post-launch"})
agg.groupby(["merchant", "cls"])[["txns", "spend"]].sum().round(0)

txns        spend
merchant          cls                             
American Airlines Link agent       11       2722.0
                  other        253606   81603644.0
Etsy              Link agent       10        629.0
                  other       1094019   50442906.0
GoDaddy           Link agent        4        194.0
                  other        164018   33451292.0
Porkbun           Link agent       10        122.0
                  other          4719     137729.0
Walmart.com       Link agent       14        921.0
                  other       7476664  499252783.0
eBay              Link agent       20       1053.0
                  other       3874044  363564785.0

In [3]:
# Link-agent share of each merchant's panel transactions and spend, before and after launch.
t = agg.groupby(["merchant", "period", "cls"])[["txns", "spend"]].sum().unstack("cls", fill_value=0)
out = pd.DataFrame({
    "txns": t["txns"].sum(axis=1),
    "link_agent_txns": t["txns"].get("Link agent", 0),
    "link_agent_txn_share_%": (100 * t["txns"].get("Link agent", 0) / t["txns"].sum(axis=1)).round(3),
    "link_agent_spend_share_%": (100 * t["spend"].get("Link agent", 0) / t["spend"].sum(axis=1)).round(3),
    "link_wallet_txn_share_%": (100 * t["txns"].get("Link wallet", 0) / t["txns"].sum(axis=1)).round(3),
})
out.sort_index(level=["merchant", "period"], ascending=[True, False])

txns  link_agent_txns  link_agent_txn_share_%  link_agent_spend_share_%  link_wallet_txn_share_%
merchant          period                                                                                                          
American Airlines pre-launch    150538                0                   0.000                     0.000                      0.0
                  post-launch   103079               11                   0.011                     0.008                      0.0
Etsy              pre-launch    647242                0                   0.000                     0.000                      0.0
                  post-launch   446787               10                   0.002                     0.003                      0.0
GoDaddy           pre-launch     95494                0                   0.000                     0.000                      0.0
                  post-launch    68528                4                   0.006                     0.001                      0.0
Porkbun           pre-launch      2630                0                   0.000                     0.000                      0.0
                  post-launch     2099               10                   0.476                     0.224                      0.0
Walmart.com       pre-launch   4445208                1                   0.000                     0.000                      0.0
                  post-launch  3031470               13                   0.000                     0.000                      0.0
eBay              pre-launch   2281104                0                   0.000                     0.000                      0.0
                  post-launch  1592960               20                   0.001                     0.001                      0.0

In [4]:
# Weekly Link-agent transactions per merchant (week starting Monday).
la = agg[agg.cls == "Link agent"].assign(week=lambda d: pd.to_datetime(d.day).dt.to_period("W-SUN").dt.start_time)
la.groupby(["week", "merchant"]).txns.sum().unstack(fill_value=0)

merchant,American Airlines,Etsy,GoDaddy,Porkbun,Walmart.com,eBay
week,,,,,,
2026-09-07,1,1,0,0,2,0
2026-09-14,2,0,0,3,1,1
2026-09-21,4,3,3,4,6,8
2026-09-28,4,6,1,3,5,11


## Conclusions (run 2026-10-08, cards complete through 2026-10-03)

**Link-agent checkout is a tiny share of spend at every top merchant except Porkbun.**

| Merchant | Panel txns after launch | Link-agent txns | Share of txns | Share of spend |
|---|---:|---:|---:|---:|
| Porkbun | 2,099 | 10 | 0.48% | 0.22% |
| American Airlines | 103,079 | 11 | 0.011% | 0.008% |
| GoDaddy | 68,528 | 4 | 0.006% | 0.001% |
| Etsy | 446,787 | 10 | 0.002% | 0.003% |
| eBay | 1,592,960 | 20 | 0.001% | 0.001% |
| Walmart.com | 3,031,470 | 13 | <0.001% | <0.001% |

- Before launch, the count was zero at every merchant except one Walmart.com transaction. After launch it grows each week: 4, 7, 28, 30 across the six merchants for the weeks from 09-07.
- **Stripe-native processing is not visible in descriptors.** No rows at these merchants carry the ordinary Link-wallet form (`LINK.COM*`). Every non-agent row uses the merchant's own descriptor, so Stripe-processed and other-processor purchases cannot be told apart.
- **Undercount.** At merchants that take Stripe directly, an agent purchase carries the merchant's normal descriptor and is invisible to the dashboard. The descriptors cannot size that; it needs a merchant-level processor list (e.g. Stripe's published customer list) or agent-side data. The dashboard's agentic count is therefore a floor that covers non-Stripe merchants only, as its title states.
- Panel counts include pending and posted copies (no member-level dedupe in this aggregate), which inflates both the numerator and the denominator similarly. The shares are read as orders of magnitude.